# Episode 0 Training Smoke Test (Notebook)

Runs a tiny episode-0 training cycle (`sdf_fc1`, `policy_value`, `fc2`) to verify
the pipeline executes end-to-end. Uses GPU if available.


In [1]:
import sys
from pathlib import Path
import torch

ROOT = Path().resolve()
if not (ROOT / 'config').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from config import Config, HyperParams
from experiments.run_utils import build_models, build_optimizers
from training.episode import Episode

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Config.DEVICE = device
print('device:', device)


device: cpu


In [2]:
# Build models and optimizers
models = build_models(device)
optimizers = build_optimizers(models)

# Small hyperparams for a quick run
hyperparams = HyperParams(
    n_samples=1000,
    n_paths=10,
    batch_size=32,
    epochs=1,
)
hyperparams.lr = 1e-3
hyperparams.min_lr = 1e-6
hyperparams.warmup_steps = 0
hyperparams.max_steps = 200


In [3]:
# Build episode 0
episode = Episode(
    models=models,
    optimizers=optimizers,
    config=Config,
    hyperparams=hyperparams,
    device=device,
    episode_id=0,
)



In [4]:
# Run episode 0 (tiny)
summary = episode.run_episode(
    n_epochs=100,
    batch_size=hyperparams.batch_size,
    log_interval=10,
    n_samples=hyperparams.n_samples,
    n_paths=hyperparams.n_paths,
    group_size=20,
    n_branches=2,
    train_modules=['sdf_fc1', 'policy_value', 'fc2'],
)
summary


Generating sdf_fc1 data:   0%|          | 0/16 [00:00<?, ?it/s]

Simulating paths: 100%|██████████| 10/10 [00:00<00:00, 176.58it/s]
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN gradient detected in fc2
NaN g

{'episode_id': 0,
 'total_steps': 1300,
 'module_summaries': {'sdf_fc1': {'final_losses': {'sdf': nan,
    'total': nan,
    'sdf_fc1_grad_norm': 64032.35438707381}},
  'policy_value': {'final_losses': {'p0': 0.23026805743575096,
    'pi': 0.23715430572628976,
    'q': 0.49858319088816644,
    'total': 0.9660055458545684,
    'policy_value_grad_norm': 1.607443242961833}},
  'fc2': {'final_losses': {'fc2': nan, 'total': nan, 'fc2_grad_norm': 0.0}}},
 'loss_history': {'sdf': [529026.1875,
   484634.875,
   442585.8125,
   406200.21875,
   372862.21875,
   343392.5625,
   316652.96875,
   290747.8125,
   265607.5,
   242141.46875,
   220217.59375,
   199216.546875,
   179056.28125,
   159820.1875,
   141566.265625,
   124075.046875,
   107347.0390625,
   91912.2265625,
   78250.609375,
   66254.4296875,
   54921.55859375,
   44318.234375,
   34994.48046875,
   26915.185546875,
   19662.486328125,
   13336.998046875,
   8033.89892578125,
   4073.484130859375,
   1327.6778564453125,
   59.8

In [8]:

for _, group in episode.df_sdf.groupby('path'):
    group = group.sort_values('branch')


In [5]:
episode.df_macro


,path,t,branch,K,C,LnK,Hatc,n_firms,M,x,hatcf,lnkf
0,0,0,-1,51.093544,4.259903,3.933658,-2.484292,20,1.000000,-2.030546,-2.310107,3.933658
1,0,1,0,61.951069,5.733899,4.126345,-2.379841,30,14.626158,-2.054311,-1.413153,2.740099
2,0,1,1,61.951069,8.207802,4.126345,-2.021184,30,14.630362,-2.043560,-1.410810,2.741183
3,1,0,-1,44.136948,7.578292,3.787297,-1.761951,20,1.000000,-1.960503,-1.761377,3.787297
4,1,1,0,54.779243,8.975187,4.003311,-1.808787,30,0.110060,-1.959668,-1.222170,2.646962
5,1,1,1,54.779243,9.308363,4.003311,-1.772339,30,0.110112,-1.973207,-1.225210,2.645236
6,2,0,-1,104.229370,15.667984,4.646594,-1.894908,20,1.000000,-1.991426,-0.451506,4.646594
7,2,1,0,116.037971,16.674940,4.753917,-1.939941,30,0.193285,-2.005269,-1.326897,3.148057
8,2,1,1,116.037971,19.892010,4.753917,-1.763541,30,0.193185,-1.995881,-1.324790,3.149310
9,3,0,-1,21.423948,4.338661,3.064509,-1.596894,20,1.000000,-1.998212,-0.522323,3.064509


In [6]:
# Inspect outputs
print('--- Episode 0 summary ---')
print(summary)

if episode.df is not None:
    print('\n[df] shape:', episode.df.shape)
    print('df columns:', episode.df.columns.tolist())
    display(episode.df.head(5))

if episode.df_macro is not None:
    print('\n[df_macro] shape:', episode.df_macro.shape)
    print('df_macro columns:', episode.df_macro.columns.tolist())
    display(episode.df_macro.head(5))

if episode.df_sdf is not None:
    print('\n[df_sdf] shape:', episode.df_sdf.shape)
    print('df_sdf columns:', episode.df_sdf.columns.tolist())
    display(episode.df_sdf.head(5))


--- Episode 0 summary ---
{'episode_id': 0, 'total_steps': 1300, 'module_summaries': {'sdf_fc1': {'final_losses': {'sdf': nan, 'total': nan, 'sdf_fc1_grad_norm': 64032.35438707381}}, 'policy_value': {'final_losses': {'p0': 0.23026805743575096, 'pi': 0.23715430572628976, 'q': 0.49858319088816644, 'total': 0.9660055458545684, 'policy_value_grad_norm': 1.607443242961833}}, 'fc2': {'final_losses': {'fc2': nan, 'total': nan, 'fc2_grad_norm': 0.0}}}, 'loss_history': {'sdf': [529026.1875, 484634.875, 442585.8125, 406200.21875, 372862.21875, 343392.5625, 316652.96875, 290747.8125, 265607.5, 242141.46875, 220217.59375, 199216.546875, 179056.28125, 159820.1875, 141566.265625, 124075.046875, 107347.0390625, 91912.2265625, 78250.609375, 66254.4296875, 54921.55859375, 44318.234375, 34994.48046875, 26915.185546875, 19662.486328125, 13336.998046875, 8033.89892578125, 4073.484130859375, 1327.6778564453125, 59.86581039428711, 198.7235870361328, 713.964599609375, 1322.53173828125, 1799.97216796875, 1979

,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_i,Bar_z,P,bp0,bpI,bp,Y,I,Phi,C
0,0,0,-1,3a4d3bab9ccc47399c9417194b2e8e28,1.0,0.141117,-0.081393,0.0,0.336798,-2.030546,...,0.123071,0.000017,0.219508,0.431409,0.355603,0.422079,0.335257,0.170209,0.000032,0.165016
1,0,0,-1,08239b2f6b734bcb924a1437653d87e9,1.0,0.070183,0.723802,0.0,0.469749,-2.030546,...,0.143930,0.000036,0.204858,0.422758,0.366106,0.414604,0.139099,0.045001,0.000014,0.094085
2,0,0,-1,da3dc499ff1744cb9304d4c081f6b936,1.0,0.423493,0.188754,0.0,0.489834,-2.030546,...,0.119137,0.000014,0.223228,0.431001,0.365721,0.423224,0.405938,0.200604,0.000025,0.205308
3,0,0,-1,e66ef65128b7427699705d9b8fa888f8,1.0,0.104618,-0.647260,0.0,0.460992,-2.030546,...,0.111752,0.000011,0.228561,0.433181,0.345568,0.423390,0.069938,0.072779,0.000007,-0.002849
4,0,0,-1,891d9b290d6f4320b0f997b69a83a9b2,1.0,0.337941,-0.795732,0.0,0.465902,-2.030546,...,0.101075,0.000007,0.238419,0.434946,0.344627,0.425817,0.223798,0.253462,0.000016,-0.029681



[df_macro] shape: (30, 12)
df_macro columns: ['path', 't', 'branch', 'K', 'C', 'LnK', 'Hatc', 'n_firms', 'M', 'x', 'hatcf', 'lnkf']


,path,t,branch,K,C,LnK,Hatc,n_firms,M,x,hatcf,lnkf
0,0,0,-1,51.093544,4.259903,3.933658,-2.484292,20,1.000000,-2.030546,-2.310107,3.933658
1,0,1,0,61.951069,5.733899,4.126345,-2.379841,30,14.626158,-2.054311,-1.413153,2.740099
2,0,1,1,61.951069,8.207802,4.126345,-2.021184,30,14.630362,-2.043560,-1.410810,2.741183
3,1,0,-1,44.136948,7.578292,3.787297,-1.761951,20,1.000000,-1.960503,-1.761377,3.787297
4,1,1,0,54.779243,8.975187,4.003311,-1.808787,30,0.110060,-1.959668,-1.222170,2.646962



[df_sdf] shape: (32, 6)
df_sdf columns: ['path', 'branch', 'x_t', 'x_t1', 'Hatcf_t', 'LnKF_t']


,path,branch,x_t,x_t1,Hatcf_t,LnKF_t
0,0,0,-1.951896,-1.979271,-2.025260,3.769603
1,0,1,-1.951896,-1.964657,-2.025260,3.769603
2,1,0,-1.983704,-1.971149,-2.065735,4.182012
3,1,1,-1.983704,-2.005338,-2.065735,4.182012
4,2,0,-2.063615,-2.057718,-1.940345,3.758796


## FC2 NaN debug (dataset + pipeline outputs)


In [7]:
# Check NaNs in episode.df (used by FC2)
import numpy as np

if episode.df is None:
    print('episode.df is None')
else:
    na_counts = episode.df.isna().sum().sort_values(ascending=False)
    print('Top NaN columns in episode.df:')
    display(na_counts.head(20))
    # quick sanity: any inf?
    num_inf = np.isinf(episode.df.select_dtypes(include=[np.number])).sum().sum()
    print('Total +inf/-inf in numeric cols:', int(num_inf))


Top NaN columns in episode.df:


path     0
Q        0
Phi      0
I        0
Y        0
bp       0
bpI      0
bp0      0
P        0
Bar_z    0
Bar_i    0
PI       0
P0       0
M        0
t        0
K        0
LnKF     0
Hatcf    0
x        0
i        0
dtype: int64

Total +inf/-inf in numeric cols: 0


In [8]:
# Run FC2LossPipe once and report tensors with NaN/Inf
import torch
from losses.FC2losspipe import FC2LossPipe

if episode.df is None:
    print('episode.df is None; skip FC2LossPipe debug')
else:
    pipe = FC2LossPipe(df=episode.df, full_N=1000, entry_num=None, device=device)
    loss_val, outputs = pipe.loss(models['fc2'], models['policy_value'])
    if isinstance(loss_val, tuple):
        loss_val = loss_val[0]
    print('fc2 loss:', loss_val)

    def _check(name, t):
        if torch.is_tensor(t):
            has_nan = torch.isnan(t).any().item()
            has_inf = torch.isinf(t).any().item()
            if has_nan or has_inf:
                try:
                    t_min = torch.nanmin(t).item()
                    t_max = torch.nanmax(t).item()
                except Exception:
                    t_min, t_max = None, None
                print(f'[NaN/Inf] {name}: nan={has_nan}, inf={has_inf}, min={t_min}, max={t_max}, shape={tuple(t.shape)}')

    for k, v in outputs.items():
        _check(k, v)


fc2 loss: tensor(nan, grad_fn=<AddBackward0>)
[NaN/Inf] PV_input: nan=True, inf=False, min=None, max=None, shape=(10, 1000, 7)
[NaN/Inf] children_s_full: nan=True, inf=False, min=None, max=None, shape=(10, 1000, 2, 5)
[NaN/Inf] FC2_input_children: nan=True, inf=False, min=None, max=None, shape=(10, 2, 201)
[NaN/Inf] FC2_output_children1: nan=True, inf=False, min=None, max=None, shape=(10, 2, 2)
[NaN/Inf] CV_input: nan=True, inf=False, min=None, max=None, shape=(10, 1000, 2, 7)
[NaN/Inf] lnk_children: nan=True, inf=False, min=None, max=None, shape=(10, 2, 1)
[NaN/Inf] hatc_children: nan=True, inf=False, min=None, max=None, shape=(10, 2, 1)
[NaN/Inf] loss_children: nan=True, inf=False, min=None, max=None, shape=()
[NaN/Inf] loss_total: nan=True, inf=False, min=None, max=None, shape=()


## FC2 pipeline step-by-step (data -> tensors -> loss pieces)
Run the episode cell first so `episode.df` and `models` exist.


In [26]:
import numpy as np
import torch
from experiments.fill_fullN_entrants import fill_df_to_fullN
from losses.FC2losspipe import FC2LossPipe

def _nan_report(name, t):
    if torch.is_tensor(t):
        has_nan = torch.isnan(t).any().item()
        has_inf = torch.isinf(t).any().item()
        if has_nan or has_inf:
            try:
                t_min = torch.nanmin(t).item()
                t_max = torch.nanmax(t).item()
            except Exception:
                t_min, t_max = None, None
            print(f'[NaN/Inf] {name}: nan={has_nan}, inf={has_inf}, min={t_min}, max={t_max}, shape={tuple(t.shape)}')
        else:
            print(f'[OK] {name}: shape={tuple(t.shape)}')

def _df_nan_report(df, name):
    na_counts = df.isna().sum().sort_values(ascending=False)
    print(f'[{name}] shape:', df.shape)
    print(f'[{name}] top NaN cols:')
    display(na_counts.head(15))
    num_inf = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
    print(f'[{name}] total inf in numeric cols:', int(num_inf))
episode.df['branch'] = episode.df['branch']+1

In [27]:
# 1) Raw df used by FC2
if episode.df is None:
    print('episode.df is None')
else:
    _df_nan_report(episode.df, 'episode.df')
    display(episode.df.head(5))


[episode.df] shape: (800, 27)
[episode.df] top NaN cols:


path     0
Q        0
Phi      0
I        0
Y        0
bp       0
bpI      0
bp0      0
P        0
Bar_z    0
Bar_i    0
PI       0
P0       0
M        0
t        0
dtype: int64

[episode.df] total inf in numeric cols: 0


,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_i,Bar_z,P,bp0,bpI,bp,Y,I,Phi,C
0,0,0,0,3a4d3bab9ccc47399c9417194b2e8e28,1.0,0.141117,-0.081393,0.0,0.336798,-2.030546,...,0.123071,0.000017,0.219508,0.431409,0.355603,0.422079,0.335257,0.170209,0.000032,0.165016
1,0,0,0,08239b2f6b734bcb924a1437653d87e9,1.0,0.070183,0.723802,0.0,0.469749,-2.030546,...,0.143930,0.000036,0.204858,0.422758,0.366106,0.414604,0.139099,0.045001,0.000014,0.094085
2,0,0,0,da3dc499ff1744cb9304d4c081f6b936,1.0,0.423493,0.188754,0.0,0.489834,-2.030546,...,0.119137,0.000014,0.223228,0.431001,0.365721,0.423224,0.405938,0.200604,0.000025,0.205308
3,0,0,0,e66ef65128b7427699705d9b8fa888f8,1.0,0.104618,-0.647260,0.0,0.460992,-2.030546,...,0.111752,0.000011,0.228561,0.433181,0.345568,0.423390,0.069938,0.072779,0.000007,-0.002849
4,0,0,0,891d9b290d6f4320b0f997b69a83a9b2,1.0,0.337941,-0.795732,0.0,0.465902,-2.030546,...,0.101075,0.000007,0.238419,0.434946,0.344627,0.425817,0.223798,0.253462,0.000016,-0.029681


In [28]:
# 2) After fill_fullN (matches FC2LossPipe init)
if episode.df is None:
    print('episode.df is None')
else:
    df_filled = fill_df_to_fullN(episode.df, full_N=1000, device=device, entry_num=None)
    _df_nan_report(df_filled, 'df_filled')
    display(df_filled.head(5))
    # Check parent rows added for new IDs
    parent_nan = df_filled[(df_filled['branch'] == 0) & (df_filled['b'].isna())]
    print('parent rows with NaN b (new IDs):', len(parent_nan))


[df_filled] shape: (30200, 28)
[df_filled] top NaN cols:


Q        29400
P0       29400
C        29400
Phi      29400
entry    29400
I        29400
Y        29400
bp       29400
bpI      29400
bp0      29400
P        29400
Bar_z    29400
Bar_i    29400
PI       29400
Entry    10600
dtype: int64

[df_filled] total inf in numeric cols: 0


,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_z,P,bp0,bpI,bp,Y,I,Phi,C,Entry
0,0,0,0,3a4d3bab9ccc47399c9417194b2e8e28,1.0,0.141117,-0.081393,0.0,0.336798,-2.030546,...,0.000017,0.219508,0.431409,0.355603,0.422079,0.335257,0.170209,0.000032,0.165016,NaN
1,0,0,0,08239b2f6b734bcb924a1437653d87e9,1.0,0.070183,0.723802,0.0,0.469749,-2.030546,...,0.000036,0.204858,0.422758,0.366106,0.414604,0.139099,0.045001,0.000014,0.094085,NaN
2,0,0,0,da3dc499ff1744cb9304d4c081f6b936,1.0,0.423493,0.188754,0.0,0.489834,-2.030546,...,0.000014,0.223228,0.431001,0.365721,0.423224,0.405938,0.200604,0.000025,0.205308,NaN
3,0,0,0,e66ef65128b7427699705d9b8fa888f8,1.0,0.104618,-0.647260,0.0,0.460992,-2.030546,...,0.000011,0.228561,0.433181,0.345568,0.423390,0.069938,0.072779,0.000007,-0.002849,NaN
4,0,0,0,891d9b290d6f4320b0f997b69a83a9b2,1.0,0.337941,-0.795732,0.0,0.465902,-2.030546,...,0.000007,0.238419,0.434946,0.344627,0.425817,0.223798,0.253462,0.000016,-0.029681,NaN


parent rows with NaN b (new IDs): 9800


In [29]:
# 3) Build FC2LossPipe and inspect internal tensors
if episode.df is None:
    print('episode.df is None')
else:
    pipe = FC2LossPipe(df=episode.df, full_N=1000, entry_num=None, device=device)
    _nan_report('P_s_full', pipe.P_s_full)
    _nan_report('K_parent_full', pipe.K_parent_full)
    _nan_report('Children_s_full', pipe.Children_s_full)
    _nan_report('K_children_full', pipe.K_children_full)
    _nan_report('alive_mask', pipe.alive_mask.float())
    _nan_report('entry_mask', pipe.entry_mask.float())


[NaN/Inf] P_s_full: nan=True, inf=False, min=None, max=None, shape=(10, 1000, 5)
[NaN/Inf] K_parent_full: nan=True, inf=False, min=None, max=None, shape=(10, 1000, 1)
[OK] Children_s_full: shape=(10, 1000, 2, 5)
[OK] K_children_full: shape=(10, 1000, 2, 1)
[OK] alive_mask: shape=(10, 1000)
[OK] entry_mask: shape=(10, 1000, 2)


In [30]:
# 4) FC2 parent input/output + PV_input
if episode.df is None:
    print('episode.df is None')
else:
    FC2_input_parent = pipe.build_fc2_input_parent()
    _nan_report('FC2_input_parent', FC2_input_parent)

    fc2_out_parent = models['fc2'](FC2_input_parent)
    _nan_report('fc2_out_parent.hatc', fc2_out_parent['hatc'])
    _nan_report('fc2_out_parent.lnk', fc2_out_parent['lnk'])

    hatc_parent_pred = fc2_out_parent['hatc']
    lnk_parent_pred = fc2_out_parent['lnk']
    FC2_output_parent = torch.cat([lnk_parent_pred, hatc_parent_pred], dim=-1)
    FC2_output_parent = FC2_output_parent.unsqueeze(1).expand(-1, pipe.full_N, -1) * pipe.alive_mask.unsqueeze(-1)

    PV_input = torch.cat((pipe.P_s_full, FC2_output_parent), dim=-1)
    _nan_report('PV_input', PV_input)


[OK] FC2_input_parent: shape=(10, 201)
[OK] fc2_out_parent.hatc: shape=(10, 1)
[OK] fc2_out_parent.lnk: shape=(10, 1)
[NaN/Inf] PV_input: nan=True, inf=False, min=None, max=None, shape=(10, 1000, 7)


In [31]:
FC2_input_parent

tensor([[ 0.0598,  0.0618,  0.0638,  ...,  1.8058,  1.9564, -2.0305],
        [ 0.0181,  0.0247,  0.0314,  ...,  1.6166,  1.7750, -1.9605],
        [ 0.0257,  0.0262,  0.0267,  ...,  1.3823,  1.3829, -1.9914],
        ...,
        [ 0.0056,  0.0114,  0.0172,  ...,  1.3415,  1.3904, -2.0306],
        [ 0.0120,  0.0161,  0.0201,  ...,  1.3914,  1.5264, -1.9629],
        [ 0.0241,  0.0249,  0.0258,  ...,  1.3567,  1.3976, -1.9726]])

In [32]:
# 5) PV forward + updated alive mask
if episode.df is None:
    print('episode.df is None')
else:
    PV_output = pipe._pv_forward(models['policy_value'], PV_input)
    bar_z = pipe._get_out(PV_output, 'bar_z', 6)
    bar_i = pipe._get_out(PV_output, 'bar_i', 5)
    bp = pipe._get_out(PV_output, 'bp', 9)
    _nan_report('bar_z', bar_z)
    _nan_report('bar_i', bar_i)
    _nan_report('bp', bp)

    alive_prob = bar_z.clamp(0, 1).squeeze(-1)
    base_alive = pipe.alive_mask.float()
    updated_alive = base_alive * alive_prob
    updated_alive = torch.nan_to_num(updated_alive, nan=0.0)
    updated_alive = updated_alive.unsqueeze(-1).expand(-1, -1, 2)
    alive_mask = updated_alive + pipe.entry_mask.float()
    _nan_report('alive_mask', alive_mask)


[OK] bar_z: shape=(10, 1000, 1)
[OK] bar_i: shape=(10, 1000, 1)
[OK] bp: shape=(10, 1000, 1)
[OK] alive_mask: shape=(10, 1000, 2)


In [33]:
bar_z

tensor([[[1.1578e-12],
         [3.7228e-13],
         [1.7516e-11],
         ...,
         [0.0000e+00],
         [0.0000e+00],
         [0.0000e+00]],

        [[4.2551e-15],
         [1.9221e-12],
         [1.7504e-13],
         ...,
         [0.0000e+00],
         [0.0000e+00],
         [0.0000e+00]],

        [[8.6330e-16],
         [5.4880e-14],
         [2.5215e-14],
         ...,
         [0.0000e+00],
         [0.0000e+00],
         [0.0000e+00]],

        ...,

        [[4.3231e-14],
         [3.3379e-15],
         [5.1425e-14],
         ...,
         [0.0000e+00],
         [0.0000e+00],
         [0.0000e+00]],

        [[2.2882e-15],
         [3.7989e-15],
         [3.2521e-13],
         ...,
         [0.0000e+00],
         [0.0000e+00],
         [0.0000e+00]],

        [[1.6304e-12],
         [1.6309e-15],
         [1.2990e-13],
         ...,
         [0.0000e+00],
         [0.0000e+00],
         [0.0000e+00]]], grad_fn=<ViewBackward0>)

In [34]:
# 6) Children path: update b, build FC2_input_children, CV_input
if episode.df is None:
    print('episode.df is None')
else:
    children_s_full = pipe.Children_s_full.clone()

    masked_b = torch.where(updated_alive[..., 0] > 0, pipe.P_s_full[:, :, 0], torch.zeros_like(pipe.P_s_full[:, :, 0]))
    masked_bp = torch.where(updated_alive[..., 0] > 0, bp[..., 0], torch.zeros_like(bp[..., 0]))
    child0 = children_s_full[:, :, 0, :]
    child1 = children_s_full[:, :, 1, :]
    new_b0 = masked_bp * child0[:, :, 2] + masked_b * (1 - child0[:, :, 2])
    new_b1 = masked_bp * child1[:, :, 2] + masked_b * (1 - child1[:, :, 2])
    child0 = torch.cat([new_b0.unsqueeze(-1), child0[:, :, 1:]], dim=-1)
    child1 = torch.cat([new_b1.unsqueeze(-1), child1[:, :, 1:]], dim=-1)
    children_s_full = torch.stack([child0, child1], dim=2)

    _nan_report('children_s_full', children_s_full)

    FC2_input_children = pipe.build_fc2_input_children(children_s_full, alive_mask)
    _nan_report('FC2_input_children', FC2_input_children)

    flat_children = FC2_input_children.reshape(-1, FC2_input_children.shape[-1])
    fc2_out_children = models['fc2'](flat_children)
    _nan_report('fc2_out_children.hatc', fc2_out_children['hatc'])
    _nan_report('fc2_out_children.lnk', fc2_out_children['lnk'])

    hatc_children_pred = fc2_out_children['hatc'].view(pipe.path_num, 2, 1)
    lnk_children_pred = fc2_out_children['lnk'].view(pipe.path_num, 2, 1)
    FC2_output_children1 = torch.cat([lnk_children_pred, hatc_children_pred], dim=-1)
    FC2_output_children = FC2_output_children1.unsqueeze(1).expand(-1, pipe.full_N, -1, -1) * alive_mask.unsqueeze(-1)

    CV_input = torch.cat((children_s_full, FC2_output_children), dim=-1)
    _nan_report('CV_input', CV_input)


[OK] children_s_full: shape=(10, 1000, 2, 5)
[OK] FC2_input_children: shape=(10, 2, 201)
[OK] fc2_out_children.hatc: shape=(20, 1)
[OK] fc2_out_children.lnk: shape=(20, 1)
[OK] CV_input: shape=(10, 1000, 2, 7)


In [35]:
children_s_full

tensor([[[[ 0.0598,  0.6252,  0.0000,  0.2449, -2.0543],
          [ 0.0598,  0.8218,  0.0000,  0.2289, -2.0436]],

         [[ 0.0702,  1.0745,  0.0000,  0.2070, -2.0543],
          [ 0.0702,  0.4882,  0.0000,  0.0513, -2.0436]],

         [[ 0.0816,  1.8522,  0.0000,  0.0404, -2.0543],
          [ 0.0816,  1.3254,  0.0000,  0.1876, -2.0436]],

         ...,

         [[ 0.0000, -0.0689,  0.0000,  0.4541, -2.0543],
          [ 0.0000,  0.0230,  0.0000,  0.3590, -2.0436]],

         [[ 0.0000, -0.6295,  0.0000,  0.0394, -2.0543],
          [ 0.0000,  0.9770,  0.0000,  0.0546, -2.0436]],

         [[ 0.0000, -0.3368,  0.0000,  0.3700, -2.0543],
          [ 0.0000,  0.1988,  0.0000,  0.0492, -2.0436]]],


        [[[ 0.0181, -0.1094,  0.0000,  0.3768, -1.9597],
          [ 0.0181,  0.0886,  0.0000,  0.2255, -1.9732]],

         [[ 0.0528,  1.5207,  0.0000,  0.2016, -1.9597],
          [ 0.0528,  1.3175,  0.0000,  0.4927, -1.9732]],

         [[ 0.1077,  1.2514,  0.0000,  0.3513, -1.9597]

In [36]:
# 7) Children loss pieces
if episode.df is None:
    print('episode.df is None')
else:
    CV_output = pipe._pv_forward(models['policy_value'], CV_input)
    bar_z_children = pipe._get_out(CV_output, 'bar_z', 6)
    bar_i_children = pipe._get_out(CV_output, 'bar_i', 5)

    alive_prob_children = bar_z_children.clamp(0, 1).squeeze(-1)
    base_alive_children = alive_mask.float()
    updated_alive_children = base_alive_children * alive_prob_children
    alive_mask_children = updated_alive_children.unsqueeze(-1)

    K_children_full = pipe.K_children_full * (1 + pipe.G * bar_i_children)

    Y = torch.exp(CV_input[..., 4:5] + CV_input[..., 1:2]) * K_children_full
    Phi = (1 - pipe.phi) * (1 + torch.exp(CV_input[..., 1:2] + CV_input[..., 3:4])) * K_children_full * bar_z_children
    I = bar_i_children * K_children_full * CV_input[..., 3:4] - bar_z_children * K_children_full + pipe.delta * K_children_full
    C = torch.abs(Y - I - Phi)

    lnk_children = torch.log(torch.sum(K_children_full * alive_mask_children[..., 0:1], dim=1) + 1e-8)
    hatc_children = torch.log(torch.sum(C * alive_mask_children[..., 0:1], dim=1) / (torch.sum(K_children_full * alive_mask_children[..., 0:1], dim=1) + 1e-8))

    _nan_report('lnk_children', lnk_children)
    _nan_report('hatc_children', hatc_children)


[OK] lnk_children: shape=(10, 2, 1)
[OK] hatc_children: shape=(10, 2, 1)


In [37]:
bar_z_children

tensor([[[[1.6809e-13],
          [3.5913e-13]],

         [[5.3221e-13],
          [1.7583e-13]],

         [[2.7537e-12],
          [9.0122e-13]],

         ...,

         [[1.5497e-14],
          [3.5310e-14]],

         [[3.1642e-15],
          [5.9253e-13]],

         [[6.4857e-15],
          [8.1003e-14]]],


        [[[1.1404e-14],
          [3.1607e-14]],

         [[1.6550e-12],
          [8.6067e-13]],

         [[6.3874e-13],
          [2.0029e-13]],

         ...,

         [[2.3467e-13],
          [7.7095e-15]],

         [[1.3565e-15],
          [1.7581e-15]],

         [[1.2779e-12],
          [7.5351e-15]]],


        [[[1.3438e-15],
          [9.9740e-16]],

         [[1.5042e-13],
          [1.8676e-14]],

         [[6.3587e-15],
          [1.7264e-14]],

         ...,

         [[2.1895e-13],
          [1.2909e-14]],

         [[1.6823e-14],
          [5.4201e-15]],

         [[1.7565e-15],
          [2.3854e-12]]],


        ...,


        [[[1.1017e-14],
          

In [19]:
pipe.df2[pipe.df2.branch == 1]

,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_z_child2,P_child2,bp0_child2,bpI_child2,bp_child2,Y_child2,I_child2,Phi_child2,C_child2,Entry_child2


In [25]:
episode.df['t'].unique()

array([0, 1])